# Fase 5: Pipeline de Inferencia

Para cerrar el proyecto con un acabado profesional de Ingeniería de Machine Learning, el último paso es asegurar la persistencia del modelo y crear un pipeline de inferencia que sea capaz de recibir datos crudos de un cliente nuevo y devolver una predicción junto con sus probabilidades.

Dado que en las fases anteriores empaquetamos todo dentro de un Pipeline de Scikit-Learn (lgbm_pipeline), tenemos una ventaja enorme: podemos guardar el preprocesador y el modelo juntos en un solo archivo.


In [1]:
import pandas as pd
import numpy as np
import re
import joblib

## Paso 1: Cargar el Pipeline


In [2]:
# Cargar el pipeline en cualquier otro script
loaded_pipeline = joblib.load("../data/models/logistic_regression_pipeline.joblib")

## Paso 2: Crear el Módulo de Inferencia (inference.py)

Para probar con nuevos registros, debemos asegurarnos de que los datos crudos entrantes pasen por las mismas reglas de limpieza de texto/regex que definimos en la Fase 1 antes de ser procesados por el pipeline.

Creamos una función de inferencia robusta:


In [3]:
# ---------------------------------------------------------
# 1. Funciones auxiliares de limpieza (Fase 1)
# ---------------------------------------------------------
def clean_numeric_string(val):
    if pd.isna(val):
        return np.nan
    cleaned = re.sub(r"[^0-9.-]", "", str(val))
    try:
        return float(cleaned) if cleaned != "" else np.nan
    except ValueError:
        return np.nan


def parse_credit_history(val):
    if pd.isna(val) or not isinstance(val, str):
        return np.nan
    years = re.search(r"(\d+)\s*Years?", val)
    months = re.search(r"(\d+)\s*Months?", val)
    total_months = 0
    if years:
        total_months += int(years.group(1)) * 12
    if months:
        total_months += int(months.group(1))
    return total_months if (years or months) else np.nan


def preprocess_raw_input(df_raw):
    """Aplica la limpieza inicial de la Fase 1 a un DataFrame de entrada."""
    df = df_raw.copy()

    # Columnas a descartar si están presentes
    cols_to_drop = ["ID", "Customer_ID", "Name", "SSN", "Month"]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # Limpieza de columnas numéricas con ruido
    noisy_cols = [
        "Age",
        "Annual_Income",
        "Num_of_Loan",
        "Num_of_Delayed_Payment",
        "Outstanding_Debt",
        "Amount_invested_monthly",
        "Monthly_Balance",
    ]
    for col in noisy_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_numeric_string)

    # Tratamiento de Credit_History_Age
    if "Credit_History_Age" in df.columns:
        df["Credit_History_Age_Months"] = df["Credit_History_Age"].apply(
            parse_credit_history
        )
        df = df.drop(columns=["Credit_History_Age"])

    return df


# ---------------------------------------------------------
# 2. Función Principal de Inferencia
# ---------------------------------------------------------
def predict_credit_risk(
    raw_data_dict, model_path="../data/models/lgbm_pipeline.joblib"
):
    """
    Recibe un diccionario con los datos en bruto de un cliente y devuelve:
    - Clase predicha (Poor, Standard, Good)
    - Probabilidades por cada categoría
    """
    # Cargar el pipeline
    pipeline = joblib.load(model_path)

    # Convertir diccionario a DataFrame
    df_input = pd.DataFrame([raw_data_dict])

    # Aplicar limpieza previa
    df_cleaned = preprocess_raw_input(df_input)

    # Mapeo de clases
    class_labels = {
        0: "Poor (Riesgo Alto)",
        1: "Standard (Riesgo Medio)",
        2: "Good (Riesgo Bajo)",
    }

    # Predicción de clase y probabilidades
    pred_class_num = pipeline.predict(df_cleaned)[0]
    pred_probs = pipeline.predict_proba(df_cleaned)[0]

    # Estructurar resultado
    result = {
        "score_predicho": class_labels[pred_class_num],
        "probabilidades": {
            "Poor": round(float(pred_probs[0]) * 100, 2),
            "Standard": round(float(pred_probs[1]) * 100, 2),
            "Good": round(float(pred_probs[2]) * 100, 2),
        },
    }

    return result

## Paso 3: Prueba con un Registro Ficticio

Probemos la función simulando la llegada de una solicitud de crédito con datos crudos (incluyendo texto ruidoso como '15\_' o "10 Years and 4 Months"):


In [4]:
# Ejemplo: Solicitud de un cliente con alto endeudamiento y retrasos de pago
nuevo_cliente = {
    "Age": "32_",  # Cadena con ruido
    "Occupation": "Scientist",
    "Annual_Income": "12500.50_",  # Cadena con ruido
    "Monthly_Inhand_Salary": 1041.70,
    "Num_Bank_Accounts": 6,
    "Num_Credit_Card": 8,
    "Interest_Rate": 25,
    "Num_of_Loan": 9,
    "Type_of_Loan": "Payday Loan, Personal Loan",
    "Delay_from_due_date": 28,
    "Num_of_Delayed_Payment": "18_",  # Cadena con ruido
    "Changed_Credit_Limit": 12.5,
    "Num_Credit_Inquiries": 12,
    "Credit_Mix": "Bad",
    "Outstanding_Debt": "4500.80_",  # Cadena con ruido
    "Credit_Utilization_Ratio": 38.5,
    "Credit_History_Age": "2 Years and 3 Months",  # Formato texto
    "Payment_of_Min_Amount": "Yes",
    "Total_EMI_per_month": 150.0,
    "Amount_invested_monthly": "50.0_",
    "Payment_Behaviour": "High_spent_Small_value_payments",
    "Monthly_Balance": 120.0,
}

# Ejecutar la inferencia
resultado = predict_credit_risk(nuevo_cliente)

# Imprimir resultado estilizado
print("=== RESULTADO DE LA EVALUACIÓN DE CRÉDITO ===")
print(f"Categoría Asignada: {resultado['score_predicho']}")
print("Probabilidades:")
for clase, prob in resultado["probabilidades"].items():
    print(f"  - {clase}: {prob}%")

=== RESULTADO DE LA EVALUACIÓN DE CRÉDITO ===
Categoría Asignada: Standard (Riesgo Medio)
Probabilidades:
  - Poor: 30.76%
  - Standard: 64.97%
  - Good: 4.27%
